# 궤적 매칭 네 모드

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.

## 패키지 설치

In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q agentevals==0.0.9 langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 환경 준비


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
print("agentevals", version("agentevals"))
print("langchain", version("langchain"))

agentevals 0.0.9
langchain 1.4.0


## 기준 궤적 준비

### 코드 읽기와 비교 경계

`create_trajectory_match_evaluator(trajectory_match_mode=...)`는 실제 궤적과 기준 궤적을 비교하는 함수를 만듭니다. 이 함수에 `outputs`와 `reference_outputs`를 넘기는 순간 평가가 실행되고 `score`를 포함한 사전이 반환됩니다. `strict`와 `unordered`는 두 궤적의 동등성을 서로 다른 순서 규칙으로 보고, `subset`과 `superset`은 실제 호출 집합을 기준 호출 집합과 어느 방향으로 포함하는지 봅니다.

포함 방향은 문장으로 확인하면 혼동이 줄어듭니다. `subset`은 실제 호출이 기준에 나온 도구 안에서만 이루어졌는지 묻기 때문에 필수 도구가 빠진 짧은 경로도 통과할 수 있습니다. `superset`은 실제 호출이 기준의 필수 도구를 모두 포함했는지 묻기 때문에 관련 없는 추가 호출도 통과할 수 있습니다. 승인 뒤 결제처럼 순서가 안전 조건이면 `strict`, 순서와 무관한 두 조회라면 `unordered`, 금지 도구 제한과 필수 도구 확인은 별도 규칙을 조합합니다.


In [3]:
# search_rules 다음 check_source가 이어지는 기준 메시지 궤적을 만듭니다.
from langchain.messages import HumanMessage, AIMessage, ToolMessage
def trace(*tools: str) -> list:
    messages=[HumanMessage("규정을 확인해 주세요.")]
    for i,name in enumerate(tools,1):
        messages += [AIMessage("", tool_calls=[{"name":name,"args":{"topic":"대출"},"id":f"c{i}","type":"tool_call"}]), ToolMessage("확인", tool_call_id=f"c{i}")]
    messages.append(AIMessage("일반 도서는 14일입니다."))
    return messages
reference=trace("search_rules", "check_source")
print([c["name"] for m in reference for c in getattr(m,"tool_calls",[])])

['search_rules', 'check_source']


## strict와 unordered 실행

`strict`는 순서까지 같아야 하고 `unordered`는 같은 호출이 있으면 순서를 무시합니다. 두 결과를 같이 보면 요구 사항이 순서에 의존하는지 판단할 수 있습니다.


In [4]:
# 두 도구 순서를 뒤집어 strict와 unordered의 순서 처리 차이를 확인합니다.
from agentevals.trajectory.match import create_trajectory_match_evaluator
reversed_trace=trace("check_source","search_rules")
strict_score=create_trajectory_match_evaluator(trajectory_match_mode="strict")(outputs=reversed_trace,reference_outputs=reference)
unordered_score=create_trajectory_match_evaluator(trajectory_match_mode="unordered")(outputs=reversed_trace,reference_outputs=reference)
print("strict", strict_score)
print("unordered", unordered_score)
assert strict_score["score"] is False and unordered_score["score"] is True

strict {'key': 'trajectory_strict_match', 'score': False, 'comment': None, 'metadata': None}
unordered {'key': 'trajectory_unordered_match', 'score': True, 'comment': None, 'metadata': None}


## subset과 superset 실행

`subset`은 실제 호출이 기준 범위 안인지, `superset`은 실제 호출이 기준의 필수 호출을 모두 포함하는지 묻습니다. 허용 누락과 허용 추가의 방향이 반대입니다.


In [5]:
# 기준보다 짧은 궤적과 추가 호출 궤적으로 포함 관계의 방향을 검증합니다.
short=trace("search_rules")
extra=trace("search_rules","check_source","audit_log")
for label,actual in [("short",short),("extra",extra)]:
    print(label)
    for mode in ["subset","superset"]:
        print(mode, create_trajectory_match_evaluator(trajectory_match_mode=mode)(outputs=actual,reference_outputs=reference)["score"])
short_subset=create_trajectory_match_evaluator(trajectory_match_mode="subset")(outputs=short,reference_outputs=reference)
extra_superset=create_trajectory_match_evaluator(trajectory_match_mode="superset")(outputs=extra,reference_outputs=reference)
print("subset은 실제 호출이 기준 호출 범위 안에 있는지, superset은 실제 호출이 기준의 필수 호출을 모두 포함하는지 봅니다.")
assert short_subset["score"] is True and extra_superset["score"] is True

short
subset True
superset False
extra
subset False
superset True
subset은 실제 호출이 기준 호출 범위 안에 있는지, superset은 실제 호출이 기준의 필수 호출을 모두 포함하는지 봅니다.
